# Pipeline Cleaning `sev.csv` -- Audit Kualitas Data Bug Severity

Notebook ini adalah versi beranotasi dari `cleaning.py` (bagian dari penelitian audit kualitas data
& klasifikasi severity laporan bug, dataset Mozilla Bugzilla, n=32.428).

**Tujuan notebook ini:** menjelaskan APA yang dibersihkan di setiap tahap, KENAPA, dan APA
DAMPAKNYA -- supaya peneliti lain (atau pembimbing/penguji) bisa memahami pipeline tanpa harus
membaca source code Python mentah.

**Sumber kebenaran (single source of truth):** `MASTER-Thesis-Bug-Severity-Data-Audit.md` (Obsidian).
Kalau ada angka yang beda dengan MASTER note, MASTER note yang benar.

**Cara pakai di Google Colab:**
1. Upload `sev.csv` ke Google Drive, atau mount Drive yang sudah berisi folder `archive/`
2. Jalankan cell demi cell dari atas ke bawah
3. Setiap tahap independen -- bisa dinyalakan/dimatikan untuk membentuk konfigurasi ablasi berbeda (RQ3)

---


## 0. Setup

Jalankan cell ini dulu di Colab untuk mount Google Drive (skip kalau jalan lokal).

In [ ]:
# ==== HANYA DI GOOGLE COLAB ====
# Uncomment 2 baris di bawah kalau notebook ini jalan di Colab dan sev.csv ada di Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Sesuaikan path ini ke lokasi folder 'archive/' berisi sev.csv
ARCHIVE_DIR = "archive"   # contoh Colab: "/content/drive/MyDrive/skripsi/archive"


In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 150)
SEVERE = {"major", "critical", "blocker"}


## 1. Muat data mentah

`sev.csv` sudah bukan data 100% mentah -- tanda baca sudah dihapus di hulu (ireversibel) dan
597 baris sudah hilang sebelum dataset ini diterima. Ini dicatat sebagai *threat to construct
validity*, bukan disembunyikan.

In [ ]:
src = Path(ARCHIVE_DIR) / "sev.csv"
df0 = pd.read_csv(src)
print(f"Baris: {len(df0)}, kolom: {list(df0.columns)}")
df0.head(2)


---
## 2. Pola Regex -- definisi "apa yang dianggap kontaminasi/noise"

Setiap regex di bawah adalah **definisi eksplisit** dari satu jenis masalah kualitas data.
Menuliskannya secara eksplisit (bukan hardcode tersembunyi di dalam fungsi) supaya bisa
diaudit dan diperdebatkan -- ini bagian dari kontribusi metodologis penelitian.

### 2.1 `POST_REPORT` -- penanda pasca-pelaporan (inti RQ1 & RQ2)

Frasa yang **mustahil ada** saat QA baru mengirim laporan bug -- karena frasa ini hanya muncul
setelah proses triase/investigasi/resolusi berjalan (status Bugzilla, komentar developer,
verifikasi QA). Contoh: `"Marking Verified"`, `"marked as a duplicate"`, `"WORKSFORME"`.

**Riwayat penting:** marker `verified` (kata tunggal) sempat dikecualikan dari sini dengan
alasan "terlalu umum/ambigu". Setelah audit manual 200 dokumen (9 Sep 2026), terbukti **100%**
kemunculan kata `verified` di dataset ini adalah komentar pasca-laporan -- bukan bahasa natural.
Sekarang wajib termasuk. Detail: MASTER note SS12 Catatan Koreksi butir 14.

In [ ]:
# Penanda pasca-pelaporan -- teks yang MUSTAHIL ada saat laporan bug baru dikirim.
# Dihapus per-BARIS, bukan per-dokumen, supaya isi laporan asli tidak ikut terbuang.
#
# SINKRONISASI WAJIB: pattern ini harus tetap identik cakupannya dengan definisi yang sama
# di verify_data.py. Sempat berbeda (lupa 'Updated by') sehingga pct_contaminated di sini
# melaporkan under-count. Lihat MASTER note SS4.6b.
POST_REPORT = re.compile(
    r"marked as a duplicate(?:\s+of\s+\d+)?"
    r"|Marking\s+(?:Verified|Duplicate|Invalid|WONTFIX|Fixed)"
    r"|WORKSFORME|worksforme|works for me"
    r"|\bRESOLVED\b"
    r"|checked in|fixed in|landed in"
    r"|In\s+reply\s+to\s+comment(?:\s+\d+)?"
    r"|Updated\s+by"
    r"|\bverified\b",
    re.IGNORECASE,
)


### 2.2 `BOILERPLATE` -- metadata Bugzilla otomatis + PII

Header seperti `"Created by X on Tuesday April 7 1998..."`, alamat email termutilasi, URL.
Ini bukan isi laporan bug, cuma metadata sistem -- tapi menyumbang banyak token ke TF-IDF
tanpa memberi sinyal tentang severity.

In [ ]:
BOILERPLATE = re.compile(
    "|".join([
        r"\b(?:Created|Updated|Reported|Modified)\s+by\s+.{0,120}?\s+on\s+"
        r"(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\s+.{0,60}?"
        r"\d{1,2}\s+\d{2}\s+\d{2}\s+(?:AM|PM)\s+[A-Z]{2,4}",
        r"\bAdditional\s+Details\b",
        r"\bComment\s+\d+\b",
        r"\bCommit\s+pushed\s+to\s+\S+\s+at\s+.{0,120}",
        # email termutilasi: "nama domain com" (tanda baca sudah hilang di hulu)
        r"\b[\w\-]+\s+(?:mozilla|netscape|gmail|yahoo|hotmail|redhat|google|apache)"
        r"\s+(?:[a-z]{2,4}\s+){0,3}(?:com|org|net|edu|uk|de)\b",
        r"\bhttps?\s+\S+(?:\s+\S+){0,10}",
        r"\bwww\s+\S+(?:\s+\S+){0,6}",
    ]),
    re.IGNORECASE,
)


### 2.3 `HIGH_CARDINALITY` -- token yang menggelembungkan vocabulary tanpa sinyal

Commit SHA, alamat memori (`0x...`), IP, angka mentah -- token unik yang muncul sekali lalu
tidak pernah lagi, membuat matriks TF-IDF sangat sparse tanpa menambah informasi prediktif.

In [ ]:
HIGH_CARDINALITY = [
    (re.compile(r"\b[0-9a-f]{20,}\b"), " "),              # commit SHA
    (re.compile(r"\b0x[0-9a-fA-F]{4,}\b"), " "),          # alamat memori
    (re.compile(r"\b\d{1,3}(?:\s+\d{1,3}){3}\b"), " "),   # IP termutilasi
    (re.compile(r"\b\d+\b"), " "),                        # angka
]


### 2.4 `FIRST_COMMENT_END` -- batas blok komentar pertama

Dipakai dekontaminasi STRUKTURAL: potong dokumen persis di titik ini, membuang SELURUH riwayat
komentar sekaligus (bukan cuma frasa penanda satu-satu). Lebih dekat ke kondisi deployment nyata
(QA kirim satu laporan baru, bukan riwayat lengkap).

In [ ]:
FIRST_COMMENT_END = re.compile(
    r"\b(?:Updated\s+by|In\s+reply\s+to\s+comment|Comment\s+\d+)\b", re.IGNORECASE
)


### 2.5 `IDENTITY` -- nama pelapor/assignee

Pelapor tertentu bisa berkorelasi dengan severity (komponen yang ditangani, senioritas) --
kalau tidak dibuang, model berisiko belajar **"siapa yang melapor"**, bukan **"apa isi bug-nya"**.

In [ ]:
IDENTITY = re.compile(
    r"\b[\w\-]+\s+(?:mozilla|netscape|gmail|yahoo|hotmail|redhat|google|apache|aol|ibm|sun)"
    r"\s+(?:[a-z]{2,4}\s+){0,3}(?:com|org|net|edu|uk|de)\b"
    r"|\b(?:Created|Updated|Reported|Modified|Assigned)\s+(?:by|to)\s+"
    r"(?:[A-Z][a-z]+\s+){1,3}",
)


### 2.6 `DATES` -- tanggal, hari, bulan, tahun, zona waktu

**Bahaya khusus:** split train/test bersifat kronologis (indeks Bugzilla mendekati urutan waktu).
Token tahun (`1998`, `2004`) jadi indikator train/test yang HAMPIR SEMPURNA pada TF-IDF --
tanpa tahap ini, model bisa "curang" belajar partisi waktu, bukan severity sungguhan.

In [ ]:
DATES = re.compile(
    r"\b(?:Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)\b"
    r"|\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\b"
    r"|\b(?:19[89]\d|20[0-2]\d)\b"
    r"|\b\d{1,2}\s+\d{2}\s+\d{2}\s+(?:AM|PM)\b"
    r"|\b[A-Z]{3}T?\b(?=\s|$)",           # zona waktu: PDT, PST, GMT
    re.IGNORECASE,
)


### 2.7 `NOISE_DOC` -- dokumen tak informatif

Dokumen yang isinya cuma `"test"`, `"ugh ok"`, `"n/a"` dst -- bukan laporan bug sungguhan,
harus disaring sebelum training.

In [ ]:
NOISE_DOC = re.compile(
    r"^(?:test\d*|testing|ignore(?:\s+this)?|verified(?:\s+\w+)?|"
    r"marking\s+\w+|closed?|fixed|wfm|dupe|duplicate|ok|ugh(?:\s+\w+)*|"
    r"n\s*a|nvm|nevermind)\s*$",
    re.IGNORECASE,
)

_WS = re.compile(r"\s+")

def _norm(t: str) -> str:
    return _WS.sub(" ", t).strip()


---
## 3. Tahap-tahap Cleaning

Setiap tahap adalah fungsi terpisah -- bisa dinyalakan/dimatikan satu per satu. Inilah yang
membuat RQ3 (ablasi cleaning: tahap mana paling berpengaruh?) bisa dijawab: jalankan konfigurasi
berbeda, bandingkan macro-F1.

**Urutan tidak boleh diubah sembarangan:**
- Dedup antar dokumen (E2) HARUS terakhir -- near-duplicate baru muncul setelah boilerplate
  dan metadata dibuang.
- Dekontaminasi (DECON/STRUCT) adalah VARIAN EKSPERIMEN, bukan default -- RQ2 membandingkan
  kondisi dengan dan tanpa tahap ini.

### Tahap A -- Struktur dasar

Buang kolom index sampah, baris kosong, normalisasi whitespace, dedup dokumen yang identik
persis. Ini tahap paling dasar -- semua konfigurasi lain dimulai dari sini.

In [ ]:
def stage_a_structure(df: pd.DataFrame) -> pd.DataFrame:
    '''A. Buang kolom index sampah, baris kosong, normalisasi whitespace, dedup persis.'''
    junk = [c for c in df.columns if c.startswith("Unnamed") or not str(c).strip()]
    df = df.drop(columns=junk)
    df["Description"] = df["Description"].astype(str)
    df = df[df["Description"].str.strip().ne("")]
    df["Description"] = df["Description"].map(_norm)
    return df.drop_duplicates(subset=["Description"]).reset_index(drop=True)


### Tahap DECON -- Dekontaminasi keyword (inti RQ2)

Buang segmen pasca-pelaporan berdasarkan pola `POST_REPORT` di atas. Dihapus per-baris (bukan
per-dokumen) supaya isi laporan asli tidak ikut hilang -- hanya frasa penandanya yang dibuang.

In [ ]:
def stage_decontaminate(df: pd.DataFrame) -> pd.DataFrame:
    '''DEKONTAMINASI. Buang segmen pasca-pelaporan (inti RQ2).'''
    df = df.copy()
    df["Description"] = df["Description"].map(lambda t: _norm(POST_REPORT.sub(" ", t)))
    return df


### Tahap STRUCT -- Dekontaminasi struktural (alternatif DECON)

Jauh lebih tegas: potong dokumen di blok komentar pertama, membuang SELURUH riwayat sekaligus
(bukan cuma frasa satu-satu). Dokumen turun drastis (~345 -> ~60 token). Kalau parse rate
(dokumen yang punya batas jelas) rendah, pendekatan ini kurang efektif -- pakai DECON sebagai
tahap utama.

In [ ]:
def stage_decon_structural(df: pd.DataFrame) -> pd.DataFrame:
    '''DEKONTAMINASI STRUKTURAL. Potong dokumen pada blok komentar pertama.'''
    def cut(t: str) -> str:
        m = FIRST_COMMENT_END.search(t)
        return _norm(t[: m.start()]) if m else t

    df = df.copy()
    matched = df["Description"].str.contains(FIRST_COMMENT_END)
    df["Description"] = df["Description"].map(cut)
    df = df[df["Description"].str.strip().ne("")].reset_index(drop=True)
    print(f"      parse rate (dokumen punya batas): {matched.mean():.1%}")
    if matched.mean() < 0.20:
        print("      PERINGATAN: parse rate < 20%. Andalkan DECON sebagai tahap utama.")
    return df


### Tahap C -- Boilerplate + PII

Buang metadata otomatis Bugzilla dan informasi identitas (nama, email, URL, timestamp).

In [ ]:
def stage_c_boilerplate(df: pd.DataFrame) -> pd.DataFrame:
    '''C. Buang boilerplate metadata Bugzilla dan PII.'''
    df = df.copy()
    df["Description"] = df["Description"].map(lambda t: _norm(BOILERPLATE.sub(" ", t)))
    return df


### Tahap H -- Buang identitas pelapor/assignee

Mencegah model belajar "siapa yang melapor" alih-alih "apa isi bug".

In [ ]:
def stage_h_identity(df: pd.DataFrame) -> pd.DataFrame:
    '''H. Buang identitas pelapor/asignee (nama, email termutilasi).'''
    df = df.copy()
    df["Description"] = df["Description"].map(lambda t: _norm(IDENTITY.sub(" ", t)))
    return df


### Tahap I -- Buang tanggal/tahun

KRITIS: split kronologis membuat token tahun jadi indikator train/test yang nyaris sempurna
pada TF-IDF. Tanpa tahap ini, model bisa mempelajari partisi waktu itu sendiri, bukan severity.

In [ ]:
def stage_i_dates(df: pd.DataFrame) -> pd.DataFrame:
    '''I. Buang tanggal, nama hari/bulan, tahun, dan zona waktu.'''
    df = df.copy()
    df["Description"] = df["Description"].map(lambda t: _norm(DATES.sub(" ", t)))
    return df


### Tahap D -- Normalisasi token bervariasi tinggi

Hash, alamat memori, angka dibuang total (bukan diganti placeholder -- placeholder tidak ada
di `vocab.lst` sehingga justru menambah OOV).

In [ ]:
def stage_d_tokens(df: pd.DataFrame) -> pd.DataFrame:
    '''D. Normalisasi token bervariasi tinggi (hash, alamat memori, angka).'''
    def norm(t: str) -> str:
        for pat, repl in HIGH_CARDINALITY:
            t = pat.sub(repl, t)
        return _norm(t)

    df = df.copy()
    df["Description"] = df["Description"].map(norm)
    return df


### Tahap E1 -- Dedup internal dokumen

Buang pengulangan blok n-gram DI DALAM satu dokumen -- artefak dari kutipan balasan berulang
(`"In reply to comment 0"` yang mengulang teks verbatim).

In [ ]:
def stage_e_internal_dedup(df: pd.DataFrame, n: int = 12) -> pd.DataFrame:
    '''E1. Buang pengulangan blok n-gram di dalam satu dokumen (artefak quote-reply).'''
    def dedupe(t: str) -> str:
        w = t.split()
        if len(w) < 2 * n:
            return t
        out, seen, i = [], set(), 0
        while i < len(w):
            gram = " ".join(w[i:i + n])
            if len(w[i:i + n]) == n and gram in seen:
                i += n
                continue
            seen.add(gram)
            out.append(w[i])
            i += 1
        return " ".join(out)

    df = df.copy()
    df["Description"] = df["Description"].map(dedupe)
    return df


### Tahap F -- Saring dokumen tak informatif

Buang dokumen seperti `"test"`, `"Ugh OK"` yang bukan laporan bug sungguhan. Ada peringatan
otomatis kalau filter ini menggerus kelas minoritas (`blocker`, `trivial`) secara tidak
proporsional -- itu tanda filter terlalu agresif.

In [ ]:
def stage_f_uninformative(df: pd.DataFrame, min_tokens: int = 5) -> pd.DataFrame:
    '''F. Saring dokumen tak informatif ('test', 'Ugh OK', dst).'''
    tok = df["Description"].str.split().str.len()
    keep = (
        tok.ge(min_tokens)
        & ~df["Description"].str.match(NOISE_DOC)
        & df["Description"].str.contains(r"[A-Za-z]{3,}", regex=True)
    )
    removed = df[~keep]
    if len(removed):
        dist = removed["Severity"].value_counts().to_dict()
        print(f"      dibuang {len(removed)} dok; sebaran severity: {dist}")
        for cls in ("blocker", "trivial"):
            n_rm = dist.get(cls, 0)
            n_all = int((df["Severity"] == cls).sum())
            if n_all and n_rm / n_all > 0.05:
                print(f"      PERINGATAN: {n_rm}/{n_all} ({n_rm / n_all:.1%}) kelas "
                      f"'{cls}' terbuang. Pertimbangkan turunkan min_tokens.")
    return df[keep].reset_index(drop=True)


### Tahap E2 -- Dedup antar dokumen (HARUS TERAKHIR)

Near-duplicate baru muncul SETELAH boilerplate dan metadata dibuang: dokumen yang tadinya
berbeda hanya di bagian metadata kini menjadi identik teksnya.

In [ ]:
def stage_e2_near_dup(df: pd.DataFrame) -> pd.DataFrame:
    '''E2. Dedup antar dokumen. HARUS dijalankan TERAKHIR.'''
    key = df["Description"].str.lower().str.replace(r"\s+", " ", regex=True)
    n_before = len(df)
    df = df[~key.duplicated()].reset_index(drop=True)
    if n_before != len(df):
        print(f"      near-duplicate dibuang: {n_before - len(df)}")
    return df


### Tahap G -- Truncate outlier panjang

Potong dokumen yang sangat panjang di kuantil ke-95 -- dokumen TIDAK dibuang seluruhnya
(bug kompleks cenderung severe, jadi membuang dokumennya = kehilangan sinyal). Panjang asli
tetap disimpan sebagai kolom terpisah karena panjang itu sendiri sering prediktif.

In [ ]:
def stage_g_truncate(df: pd.DataFrame, quantile: float = 0.95) -> pd.DataFrame:
    '''G. Potong outlier panjang. Dokumen TIDAK dibuang -- bug kompleks cenderung severe.'''
    tok = df["Description"].str.split().str.len()
    cap = int(tok.quantile(quantile))
    df = df.copy()
    df["n_tokens_original"] = tok
    df["Description"] = df["Description"].map(lambda t: " ".join(t.split()[:cap]))
    print(f"      cap = {cap} token (kuantil {quantile})")
    return df


---
## 4. Konfigurasi Ablasi

Setiap konfigurasi = kombinasi tahap yang dijalankan berurutan. Membandingkan hasil antar
konfigurasi menjawab RQ2 (dampak dekontaminasi) dan RQ3 (tahap mana paling berpengaruh).

In [ ]:
STAGES = {
    "A": ("struktur dasar", stage_a_structure),
    "DECON": ("dekontaminasi keyword", stage_decontaminate),
    "STRUCT": ("dekontaminasi struktural", stage_decon_structural),
    "C": ("boilerplate + PII", stage_c_boilerplate),
    "H": ("buang identitas pelapor", stage_h_identity),
    "I": ("buang tanggal/tahun", stage_i_dates),
    "D": ("normalisasi token", stage_d_tokens),
    "E1": ("dedup internal dokumen", stage_e_internal_dedup),
    "F": ("saring tak informatif", stage_f_uninformative),
    "E2": ("dedup antar dokumen", stage_e2_near_dup),
    "G": ("truncate outlier", stage_g_truncate),
}

CONFIGS = {
    # Titik nol: data apa adanya. CATATAN: ini BUKAN "data mentah" murni --
    # tanda baca sudah dihapus di hulu dan 597 baris sudah dibuang sebelum diterima.
    "raw":          [],
    "baseline":     ["A"],
    # --- ablasi satu-tahap: tiap baris menjawab "tahap ini berpengaruh berapa?" (RQ3)
    "plus_decon":   ["A", "DECON"],                                  # <- RQ2 (keyword)
    "plus_struct":  ["A", "STRUCT"],                                 # <- RQ2 (struktural)
    "plus_boiler":  ["A", "C"],
    "plus_identity": ["A", "H"],
    "plus_dates":   ["A", "I"],
    "plus_tokens":  ["A", "D"],
    "plus_dedup":   ["A", "E1", "E2"],
    "plus_filter":  ["A", "F"],
    "plus_trunc":   ["A", "G"],
    # --- konfigurasi lengkap
    "full_nodecon": ["A", "C", "H", "I", "D", "E1", "F", "E2", "G"],           # pembanding RQ2
    "full":         ["A", "DECON", "C", "H", "I", "D", "E1", "F", "E2", "G"],  # pipeline lengkap
    "full_struct":  ["A", "STRUCT", "C", "H", "I", "D", "E1", "F", "E2", "G"], # dekon paling tegas
}

print("Konfigurasi tersedia:")
for k, v in CONFIGS.items():
    print(f"  {k:<14} {' -> '.join(v) if v else '(tanpa tahap)'}")


### Runner -- jalankan satu konfigurasi dan lihat statistiknya

Cetak: jumlah baris & rata-rata token sebelum/sesudah tiap tahap, lalu statistik akhir
(distribusi Severity, baseline majority-class, %kontaminasi tersisa).

In [ ]:
def run_config(df0: pd.DataFrame, name: str, stages: list) -> tuple:
    print(f"\n[{name}] tahap: {' -> '.join(stages) if stages else '(tidak ada)'}")
    df = df0.copy()
    if not stages:
        junk = [c for c in df.columns if c.startswith("Unnamed") or not str(c).strip()]
        df = df.drop(columns=junk)
        df["Description"] = df["Description"].astype(str)

    for s in stages:
        label, fn = STAGES[s]
        n0, t0 = len(df), df["Description"].str.split().str.len().mean()
        df = fn(df)
        t1 = df["Description"].str.split().str.len().mean()
        print(f"    {s:<6} {label:<32} baris {n0:>6} -> {len(df):<6} "
              f"token/dok {t0:>6.0f} -> {t1:.0f}")

    tok = df["Description"].str.split().str.len()
    counts = df["Severity"].value_counts().to_dict()
    total = len(df)
    top = max(counts, key=counts.get)
    prec = counts[top] / total
    f1_base = (2 * prec / (prec + 1)) / len(counts)

    stats = {
        "config": name,
        "stages": "+".join(stages) if stages else "-",
        "n_rows": total,
        "tokens_mean": round(float(tok.mean()), 1),
        "tokens_median": int(tok.median()),
        "tokens_p99": int(tok.quantile(0.99)),
        "vocab_size": len(set(w for t in df["Description"] for w in t.split())),
        "pct_contaminated": round(100 * df["Description"].str.contains(POST_REPORT).mean(), 2),
        "n_trivial": counts.get("trivial", 0),
        "n_blocker": counts.get("blocker", 0),
        "baseline_acc": round(prec, 4),
        "baseline_macro_f1": round(f1_base, 4),
    }
    return df, stats


## 5. Contoh: bandingkan `raw` vs `plus_decon` (RQ2)

Ini menjawab pertanyaan inti RQ2: seberapa besar dampak dekontaminasi terhadap teks?

**Angka acuan yang sudah divalidasi (MASTER note SS4.6b, 9 Sep 2026):**

| Config | Kontaminasi |
|---|---:|
| `raw` | 86,39% |
| `plus_decon` | 0,74% (tervalidasi silang dengan pengukuran independen: 0,61%) |

In [ ]:
df_raw, stats_raw = run_config(df0, "raw", CONFIGS["raw"])
df_decon, stats_decon = run_config(df0, "plus_decon", CONFIGS["plus_decon"])

comparison = pd.DataFrame([stats_raw, stats_decon])
comparison[["config", "n_rows", "tokens_mean", "vocab_size", "pct_contaminated"]]


### Contoh konkret: satu dokumen sebelum dan sesudah dekontaminasi

In [ ]:
idx = 0
print("SEBELUM (raw):")
print(df_raw.loc[idx, "Description"][:400])
print()
print("SESUDAH (plus_decon):")
print(df_decon.loc[idx, "Description"][:400])


## 6. Jalankan semua konfigurasi & simpan hasil (opsional, ~beberapa menit)

Hasilkan `cleaning_analysis.csv` (bahan Tabel 4 di artikel) dan 14 file `cleaned_<config>.csv`.

In [ ]:
RUN_ALL = False  # ubah jadi True kalau mau generate semua 14 konfigurasi

if RUN_ALL:
    outdir = Path("out")
    outdir.mkdir(exist_ok=True)
    rows = []
    for name, stages in CONFIGS.items():
        df, stats = run_config(df0, name, stages)
        df.to_csv(outdir / f"cleaned_{name}.csv", index=False)
        rows.append(stats)
    analysis = pd.DataFrame(rows)
    analysis.to_csv(outdir / "cleaning_analysis.csv", index=False)
    print(analysis.to_string(index=False))
else:
    print("RUN_ALL=False -- lewati. Ubah ke True untuk generate semua konfigurasi.")


---
## 7. Ringkasan untuk pembaca cepat

| Tahap | Apa yang dihapus | Kenapa |
|---|---|---|
| A | Kolom sampah, baris kosong, duplikat persis | Struktur dasar wajib |
| DECON | Frasa pasca-pelaporan (Marking Verified, dkk) | Inti RQ1/RQ2 -- kontaminasi temporal |
| STRUCT | Seluruh blok komentar setelah yang pertama | Alternatif DECON, lebih tegas |
| C | Metadata Bugzilla otomatis, PII | Bukan isi laporan bug |
| H | Nama pelapor/assignee | Cegah model belajar "siapa" bukan "apa" |
| I | Tanggal/tahun | Cegah kebocoran lewat split kronologis |
| D | Hash, alamat memori, angka | Vocabulary bengkak tanpa sinyal |
| E1 | Pengulangan blok teks dalam 1 dokumen | Artefak kutipan balasan |
| F | Dokumen tak informatif | Bukan laporan bug sungguhan |
| E2 | Dokumen duplikat antar baris | HARUS setelah C/H/I supaya near-dup baru ketemu |
| G | Ekor distribusi panjang (>p95) | Outlier ekstrem tanpa buang dokumen |

**Untuk detail angka final, threats to validity, dan kronologi koreksi metodologi:**
lihat `MASTER-Thesis-Bug-Severity-Data-Audit.md` di Obsidian vault -- itu single source of truth.